<a href="https://colab.research.google.com/github/Lucasdz-7/Projeto-Integrador-4/blob/main/C%C3%B3digoAPI4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
import os
from google.colab import files
from google.colab import drive
pd.set_option('display.max_columns', None)

In [2]:
drive.mount('/content/drive')
origem = '/content/drive/MyDrive/Data/DadosAPI4/'

ESTRUTURA = origem + 'Estrutura Organizacional.csv'
PROD = origem + 'PROD.csv'
RES43_Essencial = origem + 'RES43_Essencial.csv'
RES43_Estrategico = origem + 'RES43_Estratégico.csv'
RES43_Normal = origem + 'RES43_Normal.csv'
CENTRO_CUSTO = origem + 'Centro de Custo.csv'
RES34 = origem + 'RES34.csv'
RES47 = origem + 'RES47.csv'
RES75 = origem + 'RES75.csv'
RES82 = origem + 'RES82.csv'
MOVIMENTACOES_MATERIAIS = origem + 'Movimentações Materiais.csv'

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
ESTRUTURA = pd.read_csv(ESTRUTURA, low_memory=False, sep=';', encoding='latin1')

PROD = pd.read_csv(PROD, low_memory=False, sep=';', encoding='latin1')

RES43_Essencial = pd.read_csv(RES43_Essencial, low_memory=False, sep=';', encoding='latin1')

RES43_Estrategico = pd.read_csv(RES43_Estrategico, low_memory=False, sep=';', encoding='latin1')

RES43_Normal = pd.read_csv(RES43_Normal, low_memory=False, sep=';', encoding='latin1')

CENTRO_CUSTO = pd.read_csv(CENTRO_CUSTO, low_memory=False, sep=';', encoding='latin1')

RES34 = pd.read_csv(RES34, low_memory=False, sep=';', encoding='latin1')

RES47 = pd.read_csv(RES47, low_memory=False, sep=';', encoding='latin1')

RES75 = pd.read_csv(RES75, low_memory=False, sep=';', encoding='latin1')

RES82 = pd.read_csv(RES82, low_memory=False, sep=';', encoding='latin1')

MOVIMENTACOES_MATERIAIS = pd.read_csv(MOVIMENTACOES_MATERIAIS, low_memory=False, sep=';', encoding='latin1')

In [4]:
RES43_Essencial['Tipo'] = 'Essencial'
RES43_Estrategico['Tipo'] = 'Estratégico'
RES43_Normal['Tipo'] = 'Normal'


RES43 = pd.concat([RES43_Essencial, RES43_Estrategico, RES43_Normal], ignore_index=True)


del RES43_Essencial, RES43_Estrategico, RES43_Normal

Criamos uma coluna com qual tipo de RES43 a linha é, depois juntamos as 3 tabelas diferentes em uma única, por fim, apagamos as 3 separadas.

In [5]:
# Função simples para arrumar os números do padrão brasileiro
def converte_numero(coluna):
    return coluna.astype(str).str.replace('.', '', regex=False).str.replace(',', '.', regex=False).astype(float)

RES34['Valor Total'] = converte_numero(RES34['Valor Total'])
RES34['Saldo Atual'] = converte_numero(RES34['Saldo Atual'])
RES34['Valor Unit'] = converte_numero(RES34['Valor Unit'])


Os valores no df RES34 não estavam padronizados.

In [6]:
# Ordenando do material mais caro para o mais barato
RES34 = RES34.sort_values(by='Valor Total', ascending=False)

# Calculando a porcentagem acumulada de cada item
RES34['% Acumulado'] = (RES34['Valor Total'].cumsum() / RES34['Valor Total'].sum()) * 100

# A regra da Curva ABC adotada será: A até 80%, B até 95%, C o resto
def classifica_abc(porcentagem):
    if porcentagem <= 80: return 'A'
    elif porcentagem <= 95: return 'B'
    else: return 'C'

# Coluna nova para classificação
RES34['Curva ABC'] = RES34['% Acumulado'].apply(classifica_abc)

display(RES34[['Cód. Red.', 'Descrição', 'Valor Total', 'Curva ABC']].head())

,Cód. Red.,Descrição,Valor Total,Curva ABC
3785,88307.0,"DORMENTE, MADEIRA PRIMEIRA CLASSE TRATADA, COM...",14588563.34,A
7923,82110.0,"RODA FERROVIARIA, TRENS SERIES 7000/7500/9500 ...",7098000.00,A
428,80493.0,"ARMACAO DE TRUQUE PARA CARROS MOTOR E REBOQUE,...",4814371.60,A
8850,15934.0,"TRILHO FERROVIARIO, PERFIL TR-57, ACO CARBONO ...",3731000.67,A
431,86844.0,"ARMARIO DE PAINEIS DO ATC, UTILIZADO NOS TRENS...",2956942.10,A


Criação das colunas referentes a curva ABC no RES34

In [7]:
# Agrupando os dados pela classe (A, B e C) e somar os valores e quantidades, cada código diferente é um item.
resumo_abc = RES34.groupby('Curva ABC').agg(
    Qtd_Itens=('Cód. Red.', 'count'),
    Valor_Total_Classe=('Valor Total', 'sum')
).reset_index()

# Somando todos itens e todo valor para criarmos sabermos a porcentagem
total_itens = resumo_abc['Qtd_Itens'].sum()
total_valor = resumo_abc['Valor_Total_Classe'].sum()

# Criando as colunas de porcentagem
resumo_abc['% Qtd Itens'] = (resumo_abc['Qtd_Itens'] / total_itens) * 100
resumo_abc['% Valor Total'] = (resumo_abc['Valor_Total_Classe'] / total_valor) * 100

 # Arredondando as casas decimais das porcentagens
resumo_abc['% Qtd Itens'] = resumo_abc['% Qtd Itens'].round(2)
resumo_abc['% Valor Total'] = resumo_abc['% Valor Total'].round(2)

# Formatando para o padrão monetário
resumo_abc['Valor_Total_Classe'] = resumo_abc['Valor_Total_Classe'].apply(
    lambda x: f'R$ {x:,.2f}'.replace(',', 'X').replace('.', ',').replace('X', '.')
)

display(resumo_abc)

,Curva ABC,Qtd_Itens,Valor_Total_Classe,% Qtd Itens,% Valor Total
0,A,556,"R$ 151.295.394,51",6.00,79.99
1,B,1448,"R$ 28.377.238,27",15.63,15.00
2,C,7262,"R$ 9.463.001,09",78.37,5.00


Tabela para dar um preview de como as informações da curva ABC deverão ficar no power bi.

In [8]:
RES34.head()

,Cód. Estr.,Cód. Red.,UN,Descrição,Status,ABNT NBR 10.004,Leilão,Doação,Reuso,Reciclagem,Log. Reversa,Aterro,Ult. Saída,Ult. Entrada,Local,Almoxarifado,Saldo Atual,Valor Unit,Valor Total,Reserva Estoque,Saldo Total,% Acumulado,Curva ABC
3785,1.03.09.08.00040-9,88307.0,un,"DORMENTE, MADEIRA PRIMEIRA CLASSE TRATADA, COM...",Ativado,NaN,NaN,NaN,NaN,NaN,NaN,NaN,17/08/2026,17/08/2026,PATIO DE DORMENTES,VIA PERMANENTE LAPA,20602.00,707.70,14588563.34,12,20614,7.713281,A
7923,1.03.09.01.00091-8,82110.0,un,"RODA FERROVIARIA, TRENS SERIES 7000/7500/9500 ...",Ativado,NaN,NaN,NaN,NaN,NaN,NaN,NaN,22/07/2025,16/07/2025,GALPÃO B1 - LAPA,MATERIAL RODANTE - LAPA,546.00,13000.00,7098000.00,0,546,11.466144,A
428,1.03.09.00.00144-2,80493.0,un,"ARMACAO DE TRUQUE PARA CARROS MOTOR E REBOQUE,...",Ativado,NaN,NaN,NaN,NaN,NaN,NaN,NaN,24/07/2026,24/07/2026,PATIO DE DORMENTES,GERAL - LAPA,16.00,300898.23,4814371.60,0,16,14.011603,A
8850,1.03.09.06.00008-0,15934.0,t,"TRILHO FERROVIARIO, PERFIL TR-57, ACO CARBONO ...",Ativado,NaN,NaN,NaN,NaN,NaN,NaN,NaN,12/08/2026,12/08/2026,PÁTIO PIRITUBA,VIA PERMANENTE LAPA,509.98,7315.99,3731000.67,0,"509,979120",15.984262,A
431,1.04.04.05.00221-5,86844.0,un,"ARMARIO DE PAINEIS DO ATC, UTILIZADO NOS TRENS...",Ativado,NaN,NaN,NaN,NaN,NaN,NaN,NaN,30/12/2025,30/12/2025,GALPAO G - PISO,GERAL - LAPA,2.00,1478471.05,2956942.10,0,2,17.547660,A


In [9]:
# Usando a função criada anteriormente para padronizar os valores do RES43
colunas_para_converter = ['Saldo', 'Consumo\nMédio\nMensal']
for col in colunas_para_converter:
    RES43[col] = converte_numero(RES43[col])

# Dividindo o consumo mensal por 30
RES43['Consumo_Diario'] = RES43['Consumo\nMédio\nMensal'] / 30.0

# Calculando os dias de cobertura, com a nova coluna de consumo_Diario e Saldo.
RES43['Dias_Cobertura'] = np.where(
    RES43['Consumo_Diario'] > 0,
    RES43['Saldo'] / RES43['Consumo_Diario'],
    0
)

# Calculando o giro de estoque com a informação de consumo médio mensal e saldo dos itens.
RES43['Giro_Anual'] = np.where(
    RES43['Saldo'] > 0,
    (RES43['Consumo\nMédio\nMensal'] * 12) / RES43['Saldo'],
    0
)

# Criando A coluna de Status de cobertura
def status_estoque(dias):
    if dias == 0: return 'Ruptura (Sem Estoque)'
    elif dias < 15: return 'Risco de Ruptura (< 15 dias)'
    elif dias > 180: return 'Excesso de Estoque (> 6 meses)'
    else: return 'Estoque Saudável'

RES43['Status Cobertura'] = RES43['Dias_Cobertura'].apply(status_estoque)

display(RES43[['CPTM', 'Descrição', 'Saldo', 'Dias_Cobertura', 'Giro_Anual', 'Status Cobertura']].head())

,CPTM,Descrição,Saldo,Dias_Cobertura,Giro_Anual,Status Cobertura
0,67490,"ABAFADOR DE RUIDO, KIT COMPOSTO DE CONCHAS, HA...",87.0,192.194404,1.873103,Excesso de Estoque (> 6 meses)
1,40,"ABRACADEIRA AUTO-TRAVANTE, MATERIAL NYLON 6.6,...",2320.0,305.934066,1.176724,Excesso de Estoque (> 6 meses)
2,98,"ABRACADEIRA TIPO D, COM CUNHA P/ APERTO (CHAVE...",360.0,332.307692,1.083333,Excesso de Estoque (> 6 meses)
3,249,"AGUARRAS, A BASE DE SOLVENTES ALIFATICOS E ARO...",171.9,282.885354,1.272600,Excesso de Estoque (> 6 meses)
4,82513,"ARRUELA DE PRESSAO, EM ACO MOLA, ACABAMENTO BI...",100.0,90.009001,3.999600,Estoque Saudável


Usamos a lógica que aprendemos na máteria de administração de materiais para criação da coluna status de cobertura, não é ideal ter um estoque com dias de cobertura muito alto

In [10]:
# Agrupando por Código do Item, Descrição e Almoxarifado
estoque_por_local = RES34.groupby(['Cód. Red.', 'Descrição', 'Almoxarifado']).agg(
    Quantidade_Fisica=('Saldo Atual', 'sum'),
    Valor_Dinheiro=('Valor Total', 'sum')
).reset_index()

# Ordenar para ver os itens com maior quantidade física primeiro
estoque_por_local = estoque_por_local.sort_values(by='Quantidade_Fisica', ascending=False)

display(estoque_por_local.head(10))

,Cód. Red.,Descrição,Almoxarifado,Quantidade_Fisica,Valor_Dinheiro
2528,14753.0,TRILHO FERROVIARIO USADO QUE NAO ATENDE AOS PA...,SUCATA-ENG.TRINDADE,495912.8,123260.12
2532,14756.0,SUCATA FERROSA DE VIA PERMANENTE PEQUENO VOLUM...,SUCATA-PIRITUBA,314606.0,104038.85
2643,15504.0,"TIREFAO, CABECA PIRAMIDAL RETANGULAR, DIAM COR...",VIA PERMANENTE LAPA,252300.0,2073442.88
2448,14064.0,SUCATA DE RODAS FERROVIARIAS,SUCATA - LAPA,179480.0,79234.19
2530,14753.0,TRILHO FERROVIARIO USADO QUE NAO ATENDE AOS PA...,SUCATA-RODA D AGUA,125566.2,31209.73
2521,14726.0,SUCATA DE ACO/FERRO MIUDA EM GERAL,SUCATA - LAPA,106742.0,93154.29
146,860.0,ANILHA PARA MARCACAO DE FIO EM LETRAS DE A A Z,PRINCIPAL - ALTINO,70666.0,424.00
7606,85996.0,"DISCO DE FREIO, INSERVIVEL",SUCATA - LAPA,67401.0,10193.62
3682,31552.0,CUPILHA DE 1/8 X 1 1/2,PRINCIPAL - ALTINO,59713.0,1194.26
8819,90506.0,"ISOLADOR PARA GRAMPO ELASTICO, TIPO OLHAL,TRIL...",VIA PERMANENTE LAPA,57544.0,84180.54


Mais um preview de como podemos usar o RES34 no power bi

In [11]:
# Convertendo a coluna de data para um formato ideal
RES47['Última Movimentação'] = pd.to_datetime(RES47['Última Movimentação'], format='%d/%m/%Y', errors='coerce')

# Usamos como referência de último dia (dia de hoje), o dia em que as tabelas foram baixadas para nos enviar.
data_hoje = pd.to_datetime('2026-08-25')

# Calculando quantos dias o material está parado
RES47['Dias_Parado'] = (data_hoje - RES47['Última Movimentação']).dt.days

# alerta de obsolescência (Foco no tempo parado / sem uso)
def alerta_obsolescencia(dias):
    if pd.isna(dias): return 'Sem Informação'
    elif dias <= 90: return 'Movimentação Recente (Até 3 meses)'
    elif dias <= 180: return 'Atenção (3 a 6 meses sem uso)'
    elif dias <= 365: return 'Uso Lento (6 meses a 1 ano parado)'
    else: return 'Risco de Obsolescência (> 1 ano parado)'

RES47['Status Obsolescência'] = RES47['Dias_Parado'].apply(alerta_obsolescencia)

for col in ['Quantidade', 'Custo Médio', 'Valor Total']:
    RES47[col] = converte_numero(RES47[col])

display(RES47[['Código Reduzido', 'Descrição', 'Dias_Parado', 'Status Obsolescência']].head())

,Código Reduzido,Descrição,Dias_Parado,Status Obsolescência
0,19,"ABRACADEIRA REGULAVEL P/ ROSCA S/ FIM, DIAM 12...",60,Movimentação Recente (Até 3 meses)
1,24,"ABRACADEIRA REGULAVEL P/ ROSCA S/ FIM, DIAM 19...",182,Uso Lento (6 meses a 1 ano parado)
2,25,"ABRACADEIRA, SIMPLES PARA TUBO OU CABO, ACO AB...",963,Risco de Obsolescência (> 1 ano parado)
3,30,"ABRACADEIRA TIPO D, BITOLA 1POL, NAO ACOMPANHA...",63,Movimentação Recente (Até 3 meses)
4,33,"ABRACADEIRA AUTO-TRAVANTE, COMP 390MM, DIAM MA...",6,Movimentação Recente (Até 3 meses)


In [12]:
RES47.head()

,Código Reduzido,Código Estruturado,Descrição,Tipo Consumo,Utilização,Unid. Medida,Quantidade,Custo Médio,Valor Total,Última Movimentação,Último Consumo,Dias_Parado,Status Obsolescência
0,19,1.03.01.00.00001-7,"ABRACADEIRA REGULAVEL P/ ROSCA S/ FIM, DIAM 12...",Normal,NaN,un,500.0,1.000000,500.000000,2026-06-26,26/06/2026,60,Movimentação Recente (Até 3 meses)
1,24,1.03.01.00.00005-5,"ABRACADEIRA REGULAVEL P/ ROSCA S/ FIM, DIAM 19...",Normal,NaN,un,38.0,1.000000,38.000000,2026-02-24,24/02/2026,182,Uso Lento (6 meses a 1 ano parado)
2,25,1.03.01.00.00006-2,"ABRACADEIRA, SIMPLES PARA TUBO OU CABO, ACO AB...",Normal,MANGUEIRAS DE OXIGENIO E PORTAS DE ACESSO DE P...,un,82.0,0.193587,15.874131,2024-01-05,05/01/2024,963,Risco de Obsolescência (> 1 ano parado)
3,30,1.03.01.00.00008-6,"ABRACADEIRA TIPO D, BITOLA 1POL, NAO ACOMPANHA...",Normal,NaN,un,2219.0,0.725002,1608.779993,2026-06-23,15/01/2026,63,Movimentação Recente (Até 3 meses)
4,33,1.03.01.00.00010-9,"ABRACADEIRA AUTO-TRAVANTE, COMP 390MM, DIAM MA...",Normal,AMARRACAO E FIXACAO DE CHICOTES E CABOS,un,2302.0,0.612605,1410.217355,2026-08-19,19/08/2026,6,Movimentação Recente (Até 3 meses)


In [13]:
# Convertendo as colunas de data para um formato ideal
MOVIMENTACOES_MATERIAIS['Data Aprovação'] = pd.to_datetime(MOVIMENTACOES_MATERIAIS['Data Aprovação'], format='%d/%m/%Y %H:%M:%S', errors='coerce')
MOVIMENTACOES_MATERIAIS['Data Entrega'] = pd.to_datetime(MOVIMENTACOES_MATERIAIS['Data Entrega'], format='%d/%m/%Y %H:%M:%S', errors='coerce')

# Quando subtraímos duas datas diferentes a biblioteca transforma o resultado em segundos, nesse caso como queremos os dias só precisamos dividir os segundos por 86400
tempo_bruto = (MOVIMENTACOES_MATERIAIS['Data Entrega'] - MOVIMENTACOES_MATERIAIS['Data Aprovação']).dt.total_seconds() / 86400.0

# Tratando falhas do ERP (entregas no mesmo dia = 0; anos irreais ou negativos = nulo)
tempo_ajustado = np.where((tempo_bruto >= -1) & (tempo_bruto < 0), 0.0, tempo_bruto)
tempo_ajustado = np.where(MOVIMENTACOES_MATERIAIS['Data Entrega'].dt.year < 2015, np.nan, tempo_ajustado)
tempo_ajustado = np.where(tempo_ajustado < 0, np.nan, tempo_ajustado)

# Guardando o tempo correto e final na tabela
MOVIMENTACOES_MATERIAIS['Tempo_Atendimento_Dias'] = tempo_ajustado

# criando regra de atraso, se o tempo de atendimento for maior que dois dias = atraso/gargalo
MOVIMENTACOES_MATERIAIS['Status Logístico'] = np.where(
    MOVIMENTACOES_MATERIAIS['Tempo_Atendimento_Dias'] > 2.0,
    'Gargalo / Atraso',
    'No Prazo'
)

display(MOVIMENTACOES_MATERIAIS[['Número', 'Data Aprovação', 'Data Entrega', 'Tempo_Atendimento_Dias', 'Status Logístico']].head())

,Número,Data Aprovação,Data Entrega,Tempo_Atendimento_Dias,Status Logístico
0,419262,2021-04-06 14:26:15,2021-04-08,1.398438,No Prazo
1,419223,2021-04-05 14:41:43,2021-04-08,2.387697,Gargalo / Atraso
2,419222,2021-04-05 14:41:46,2021-04-08,2.387662,Gargalo / Atraso
3,423028,2021-07-29 15:08:33,2021-07-30,0.369063,No Prazo
4,429339,2022-02-11 11:30:05,2022-02-14,2.520775,Gargalo / Atraso


  1 hora = 60 minutos (60 x 60 = 3.600 segundos)
  1 dia = 24 horas (24 x 3.600 = 86.400 segundos)

   se quiséssemos apenas oo número de dias redondos (ignorando completamente as horas de diferença), o Pandas tem um atalho rápido, basta utilizar .dt.days em vez de .dt.total_seconds().

   Vários valores estranhos como números negativos apareceram quando ainda não tínhamos esse tratamento de falhas.

In [14]:
# Se for necessário rodarmos esse bloco mais de uma vez no mesmo ambiente de execução, esse código impede que ocorra um erro.
cols_limpar = [c for c in RES34.columns if 'Status' in c or 'Crítico' in c]
RES34 = RES34.drop(columns=cols_limpar, errors='ignore')

# Padronizando os códigos, espaços, zeros à esquerda e casas decimais
RES34['Cod_Padrao'] = RES34['Cód. Red.'].astype(str).str.split('.').str[0].str.strip().str.lstrip('0')
PROD['Cod_Padrao'] = PROD['Reduzido'].astype(str).str.split('.').str[0].str.strip().str.lstrip('0')

# Merge com as duas tabelas
prod_info = PROD[['Cod_Padrao', 'Status', 'Crítico']].drop_duplicates(subset=['Cod_Padrao'])
RES34 = RES34.merge(prod_info, on='Cod_Padrao', how='left')

# 4. Preencher espaços vazios
RES34['Status'] = RES34['Status'].fillna('Não Informado')
RES34['Crítico'] = RES34['Crítico'].fillna('Não')

PROD['Status'] = PROD['Status'].fillna('Não Informado')
PROD['Crítico'] = PROD['Crítico'].fillna('Não')
PROD['Grupo do Produto'] = PROD['Grupo do Produto'].fillna('Sem Grupo')
PROD['Nome'] = PROD['Nome'].fillna('Sem Descrição')
PROD['Estruturado'] = PROD['Estruturado'].fillna('Sem Código Estruturado')

CENTRO_CUSTO['Sigla'] = CENTRO_CUSTO['Sigla'].fillna('Sem Sigla')
CENTRO_CUSTO['Linha'] = CENTRO_CUSTO['Linha'].fillna('Linha Não Informada').str.strip()
CENTRO_CUSTO['Nome'] = CENTRO_CUSTO['Nome'].fillna('Centro de Custo Desconhecido')

ESTRUTURA['Sigla CC'] = ESTRUTURA['Sigla CC'].fillna('Sem Sigla')
ESTRUTURA['LINHA'] = ESTRUTURA['LINHA'].fillna('Sem Linha')
ESTRUTURA['DESCRIÇÃO LINHA'] = ESTRUTURA['DESCRIÇÃO LINHA'].fillna('Sem Descrição')

In [15]:
# TESTE PARA CRIAÇÃO DO GRÁFICO DE SERRA

# Padronizando essa coluna com função criada no início do código
RES43['Quantidade Necessária p/ Compra'] = converte_numero(RES43['Quantidade Necessária p/ Compra'])

tempo_reposicao_dias = 30
dias_seguranca = 15

# Calculando o Estoque de Segurança, consumo diário * Ddas de degurança
RES43['Estoque_Seguranca'] = RES43['Consumo_Diario'] * dias_seguranca

# Calculando o Ponto de Pedido, (Consumo Diário * Lead Time) + Estoque de Segurança
RES43['Ponto_Pedido'] = (RES43['Consumo_Diario'] * tempo_reposicao_dias) + RES43['Estoque_Seguranca']

# Definindo o Lote de Compra, se o ERP tiver o valor usamos ele. Se for zero, calculamos 3 meses de estoque por padrão
RES43['Lote_Compra'] = np.where(
    RES43['Quantidade Necessária p/ Compra'] > 0,
    RES43['Quantidade Necessária p/ Compra'],
    RES43['Consumo\nMédio\nMensal'] * 3
)

# Arredondando os valores para não termos "meia peça" de material
colunas_arredondar = ['Estoque_Seguranca', 'Ponto_Pedido', 'Lote_Compra']
for col in colunas_arredondar:
    RES43[col] = np.ceil(RES43[col]) # Arredonda sempre para cima

display(RES43[['CPTM', 'Descrição', 'Consumo_Diario', 'Estoque_Seguranca', 'Ponto_Pedido', 'Lote_Compra']].head(10))

,CPTM,Descrição,Consumo_Diario,Estoque_Seguranca,Ponto_Pedido,Lote_Compra
0,67490,"ABAFADOR DE RUIDO, KIT COMPOSTO DE CONCHAS, HA...",0.452667,7.0,21.0,76.0
1,40,"ABRACADEIRA AUTO-TRAVANTE, MATERIAL NYLON 6.6,...",7.583333,114.0,342.0,410.0
2,98,"ABRACADEIRA TIPO D, COM CUNHA P/ APERTO (CHAVE...",1.083333,17.0,49.0,30.0
3,249,"AGUARRAS, A BASE DE SOLVENTES ALIFATICOS E ARO...",0.607667,10.0,28.0,47.0
4,82513,"ARRUELA DE PRESSAO, EM ACO MOLA, ACABAMENTO BI...",1.111000,17.0,50.0,300.0
5,82513,"ARRUELA DE PRESSAO, EM ACO MOLA, ACABAMENTO BI...",1.111000,17.0,50.0,300.0
6,1314,"AVENTAL DE RASPA DE COURO CURTIDA AO CROMO, CO...",0.011000,1.0,1.0,4.0
7,40857,"BATERIA BOTAO; CR2032; 3V; LITIO, ECOLOGICAMEN...",0.639000,10.0,29.0,58.0
8,1775,"BLOCO VAZADO DE CONCRETO, MEDINDO (14X19X39)CM...",0.833333,13.0,38.0,300.0
9,2114,"BOTA DE SEGURANCA, SEM BIQUEIRA DE ACO, TAM 46...",0.011000,1.0,1.0,4.0


Como as bases do ERP não trouxeram o lead time dos fornecedores nem a política de segurança, nós parametrizamos o código com 30 dias de reposição (lead time) e 15 dias de segurança para provar que a lógica do Ponto de Pedido funciona. Se a CPTM inserir esses dados reais no sistema, o nosso código e o Power BI já absorvem e recalculam tudo automaticamente.

In [16]:
# 1. Função para limpar zeros à esquerda e casas decimais dos códigos
def padronizar_codigo(serie):
    return serie.astype(str).str.split('.').str[0].str.strip().str.lstrip('0')

    # 2. Padronização das chaves em todas as tabelas
PROD['Cod_Padrao'] = padronizar_codigo(PROD['Reduzido'])
RES34['Cod_Padrao'] = padronizar_codigo(RES34['Cód. Red.'])
RES43['Cod_Padrao'] = padronizar_codigo(RES43['CPTM'])
RES47['Cod_Padrao'] = padronizar_codigo(RES47['Código Reduzido'])

In [17]:
# Limpeza: Fato_Estoque_ABC (RES34)
cols_res34 = [
    'ABNT NBR 10.004', 'Leilão', 'Doação', 'Reuso',
    'Reciclagem', 'Log. Reversa', 'Aterro',
    'Cód. Estr.', 'Cód. Red.', 'UN', 'Reduzido'
]
RES34 = RES34.drop(columns=cols_res34, errors='ignore')

# Limpeza: Fato_Giro_Cobertura (RES43)
cols_res43 = ['Seq.', 'CPTM', 'Cód.Estr.', 'Código\nBEC', 'Classe BEC', 'Unnamed: 7']
RES43 = RES43.drop(columns=cols_res43, errors='ignore')

# Limpeza: Fato_Obsolescencia (RES47)
cols_res47 = ['Código Reduzido', 'Código Estruturado', 'Unid. Medida']
RES47 = RES47.drop(columns=cols_res47, errors='ignore')

# Limpeza: Dim_Produtos (PROD)
# O 'Reduzido' original já não é necessário, pois criamos o 'Cod_Padrao'
cols_prod = ['Reduzido']
PROD = PROD.drop(columns=cols_prod, errors='ignore')

# Limpeza: Fato_Eficiencia_Logistica (MOVIMENTACOES)
# Como não vamos analisar a produtividade de cada separador/entregador, também incluímos os nomes dos funcionários na lista de exclusão para economizar muita memória.
cols_mov = [
    'Equipamento', 'Sucata', 'Funcionário Entregou',
    'Funcionário', 'Funcionário Separação', 'Funcionário Retirou'
]
MOVIMENTACOES_MATERIAIS = MOVIMENTACOES_MATERIAIS.drop(columns=cols_mov, errors='ignore')

# 6. Limpeza: Dim_Centro_Custo
cols_cc = ['Inicial', 'Final']
CENTRO_CUSTO = CENTRO_CUSTO.drop(columns=cols_cc, errors='ignore')

Muitas tabelas possuem colunas que não serão utilizadas para o propósito do projeto


In [18]:
# Criando um intervalo contínuo de datas diárias de 2020 a 2026
intervalo_datas = pd.date_range(start='2020-01-01', end='2026-12-31', freq='D')

# Construindo o DataFrame da Dimensão Calendário
dim_calendario = pd.DataFrame({'Data': intervalo_datas})

# Criando as colunas com dimensões de data mais utilizadas
dim_calendario['Ano'] = dim_calendario['Data'].dt.year
dim_calendario['Mes_Num'] = dim_calendario['Data'].dt.month
dim_calendario['Mes_Nome'] = dim_calendario['Data'].dt.strftime('%B')
dim_calendario['Ano_Mes'] = dim_calendario['Data'].dt.strftime('%Y-%m')
dim_calendario['Trimestre'] = 'T' + dim_calendario['Data'].dt.quarter.astype(str)
dim_calendario['Trimestre_Ano'] = dim_calendario['Data'].dt.quarter.astype(str) + 'T/' + dim_calendario['Ano'].astype(str)
dim_calendario['Dia'] = dim_calendario['Data'].dt.day
dim_calendario['Dia_Semana'] = dim_calendario['Data'].dt.strftime('%A')
dim_calendario['Dia_Semana_Num'] = dim_calendario['Data'].dt.dayofweek + 1  # 1=Segunda, 7=Domingo

# Formatando a coluna Data para o padrão do Brasil
dim_calendario['Data_Formatada'] = dim_calendario['Data'].dt.strftime('%d/%m/%Y')

# Dicionários de tradução assim como o utilizado na API 3
meses_pt = {
    1: 'Janeiro', 2: 'Fevereiro', 3: 'Março', 4: 'Abril',
    5: 'Maio', 6: 'Junho', 7: 'Julho', 8: 'Agosto',
    9: 'Setembro', 10: 'Outubro', 11: 'Novembro', 12: 'Dezembro'
}

dias_pt = {
    1: 'Segunda-feira', 2: 'Terça-feira', 3: 'Quarta-feira',
    4: 'Quinta-feira', 5: 'Sexta-feira', 6: 'Sábado', 7: 'Domingo'
}

# Aplicar a tradução nas colunas
dim_calendario['Mes_Nome'] = dim_calendario['Mes_Num'].map(meses_pt)
dim_calendario['Dia_Semana'] = dim_calendario['Dia_Semana_Num'].map(dias_pt)

# Versões abreviadas
dim_calendario['Mes_Abrev'] = dim_calendario['Mes_Nome'].str[:3]
dim_calendario['Dia_Semana_Abrev'] = dim_calendario['Dia_Semana'].str[:3]

In [19]:
# Definindo pasta de destino no Google Drive
pasta_destino = '/content/drive/MyDrive/Arquivos 4° Semestre/'
os.makedirs(pasta_destino, exist_ok=True)

print("Salvando os arquivos CSV tratados...")

# Salvando Tabelas de Indicadores
RES34.to_csv(pasta_destino + 'Fato_Estoque_ABC.csv', index=False, sep=';', decimal=',', encoding='utf-8-sig')
RES43.to_csv(pasta_destino + 'Fato_Giro_Cobertura.csv', index=False, sep=';', decimal=',', encoding='utf-8-sig')
RES47.to_csv(pasta_destino + 'Fato_Obsolescencia.csv', index=False, sep=';', decimal=',', encoding='utf-8-sig')
MOVIMENTACOES_MATERIAIS.to_csv(pasta_destino + 'Fato_Eficiencia_Logistica.csv', index=False, sep=';', decimal=',', encoding='utf-8-sig')

# Salvando Tabelas de Dimensão (Cadastros e Filtros)
colunas_prod = ['Cod_Padrao', 'Reduzido', 'Estruturado', 'Nome', 'Grupo do Produto', 'Status', 'Crítico']
colunas_existentes = [col for col in colunas_prod if col in PROD.columns]
PROD[colunas_existentes].to_csv(pasta_destino + 'Dim_Produtos.csv', index=False, sep=';', decimal=',', encoding='utf-8-sig')
dim_calendario.to_csv(pasta_destino + 'Dim_Calendario.csv', index=False, sep=';', encoding='utf-8-sig')
ESTRUTURA.to_csv(pasta_destino + 'Dim_Estrutura_Org.csv', index=False, sep=';', decimal=',', encoding='utf-8-sig')
CENTRO_CUSTO.to_csv(pasta_destino + 'Dim_Centro_Custo.csv', index=False, sep=';', decimal=',', encoding='utf-8-sig')

print("Arquivos salvos no Drive com sucesso! Compactando...")

# Compactar os arquivos gerados e iniciar o download
comando_zip = f"zip -j /content/Bases_Tratadas_CPTM.zip '{pasta_destino}'*.csv"
os.system(comando_zip)

files.download('/content/Bases_Tratadas_CPTM.zip')
print("Download iniciado!")

Salvando os arquivos CSV tratados...
Arquivos salvos no Drive com sucesso! Compactando...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Download iniciado!
